# Introduction

We are now moving to the final part of the workshop, which involves formulating business recommendations. Our tasks are:
- Determining a global betting odds,
- Dividing the dataset into categories: A, B, C, D, where A is the best group and D is the weakest group,
- Determining the risk of odds based on accepted parameters for each category.

As the last task, in a discussion format, we must consider the fact that we are a new betting company. When formulating our recommendations, we need to identify the risks that may affect our operations. We will perform this task together in a brainstorming session.

# Notebook Configuration

## Import necessary libraries

In [124]:
import pandas as pd

## Loading data into the workspace

> Remember to correctly specify the column separator

In [77]:
df = pd.read_csv('../data/processed/hockey_teams.csv', sep=";")

### Checking data loading accuracy

In [125]:
df.head(5)

,team,season,victories,defeats,overtime_defeats,victory_percentage,scored_goals,received_goal,goal_difference,goals_ratio,league
0,Colorado Avalanche,2003,40,22,7,0.488,236,198,38,1.191919,C
1,Dallas Stars,2003,41,26,2,0.500,194,175,19,1.108571,C
2,Detroit Red Wings,2003,48,21,2,0.585,255,189,66,1.349206,B
3,Edmonton Oilers,2003,36,29,5,0.439,221,208,13,1.062500,C
4,Florida Panthers,2003,28,35,4,0.341,188,221,-33,0.850679,D


# Determining Betting Odds

Let's review the content of the page: [click](https://trustbet.pl/kursy-bukmacherskie/), where information about methods for determining betting odds can be found. First, we will determine a global odd, which will be the starting point for our analysis (the so-called _baseline scenario_). At this point, we ignore the margin and assume that we are calculating the decimal odd.

Here is the list of steps to be performed to obtain the desired value:
- we will complete the definition of the `get_betting_odds` function, which will take `probability` of a given event as a parameter. We will use it multiple times, so it is worth preparing its implementation now
- then we need to appropriately aggregate the set and determine the **global** probability of the team's victory.

## Implementations of the `get_betting_odds` function

In [79]:
def get_betting_odds(probability):
    return 1/probability

### Some tests to check the correctness of the implementation

In [80]:
def test_get_betting_odds():
    assert get_betting_odds(1) == 1, "Expected 1"
    assert get_betting_odds(0.5) == 2, "Expected 2"
    assert get_betting_odds(0.25) == 4, "Expected 4"
    assert get_betting_odds(0.1) == 10, "Expected 10"
    try:
        get_betting_odds(0)
    except ZeroDivisionError:
        pass
    else:
        assert False, "Expected ZeroDivisionError"

    print("All tests passed!")

test_get_betting_odds()

All tests passed!


### Determining the global odds

Here, determine the probability of any team winning

In [126]:
wins_and_losses = df['victories'].sum() + df['defeats'].sum()

win_prob = df['victories'].sum() / wins_and_losses
win_prob_rounded = round(win_prob, 4)
win_prob_rounded

np.float64(0.5331)

Set the global rate here using the `get_betting_odds` function. Round the result to two decimal places.

In [127]:
global_rate = get_betting_odds(win_prob_rounded)
global_rate_rounded = round(global_rate, 2)
print(f"Global rate:", global_rate_rounded)

Global rate: 1.88


# Team Categorization

Let's discuss how we can classify teams into _leagues_. We want to establish 4 leagues:
- A - league consisting of the best teams,
- B - league consisting of good teams,
- C - league consisting of average teams,
- D - league consisting of the weakest teams.

The above terms are quite subjective, so for the purpose of this exercise, we will adopt the following assumptions:
- A - the top 5% of teams,
- B - teams performing better than 70% of the group but worse than league A,
- C - teams performing better than 20% of the group but worse than league B,
- D - the remaining teams.

To accomplish this task, we will additionally implement the function `assign_team_to_league`.

> Note: This task looks unassuming, but it is difficult. Remember that during the class, you have access to the instructor, and later to a mentor.

## Determination of cutoff points for individual leagues

In [93]:
worst_vp = df['victory_percentage'].min()
print("Worst victory percentage:", worst_vp)
best_vp = df['victory_percentage'].max()
print("Best victory percentage:",best_vp)

league_A_edge = round(df['victory_percentage'].quantile(0.95), 3)
league_B_edge = df['victory_percentage'].quantile(0.7)
league_C_edge = df['victory_percentage'].quantile(0.2)
print(f"The bottom edge of league A:", league_A_edge)
print(f"The bottom edge of league B:", league_B_edge)
print(f"The bottom edge of league C:", league_C_edge)

Worst victory percentage: 0.119
Best victory percentage: 0.756
The bottom edge of league A: 0.622
The bottom edge of league B: 0.512
The bottom edge of league C: 0.366


In [94]:
def assign_team_to_league(x):
    if x >= league_A_edge:
        return "A"
    elif x >= league_B_edge:
        return "B"
    elif x >= league_C_edge:
        return "C"
    else:
        return "D"

In [95]:
df['league'] = df['victory_percentage'].apply(assign_team_to_league)

In [102]:
df.head(5)

,team,season,victories,defeats,overtime_defeats,victory_percentage,scored_goals,received_goal,goal_difference,goals_ratio,league
0,Colorado Avalanche,2003,40,22,7,0.488,236,198,38,1.191919,C
1,Dallas Stars,2003,41,26,2,0.500,194,175,19,1.108571,C
2,Detroit Red Wings,2003,48,21,2,0.585,255,189,66,1.349206,B
3,Edmonton Oilers,2003,36,29,5,0.439,221,208,13,1.062500,C
4,Florida Panthers,2003,28,35,4,0.341,188,221,-33,0.850679,D


In [99]:
df.to_csv('../data/processed/hockey_teams_leagues.csv',sep=';',index=False)

In [100]:
df = pd.read_csv('../data/processed/hockey_teams_leagues.csv', sep=";")

In [103]:
df.head(5)

,team,season,victories,defeats,overtime_defeats,victory_percentage,scored_goals,received_goal,goal_difference,goals_ratio,league
0,Colorado Avalanche,2003,40,22,7,0.488,236,198,38,1.191919,C
1,Dallas Stars,2003,41,26,2,0.500,194,175,19,1.108571,C
2,Detroit Red Wings,2003,48,21,2,0.585,255,189,66,1.349206,B
3,Edmonton Oilers,2003,36,29,5,0.439,221,208,13,1.062500,C
4,Florida Panthers,2003,28,35,4,0.341,188,221,-33,0.850679,D


In [128]:
league_A = df[df['league'] == "A"]['team']
league_A_number = len(league_A)
league_B = df[df['league'] == "B"]['team']
league_B_number = len(league_B)
league_C = df[df['league'] == "C"]['team']
league_C_number = len(league_C)
league_D = df[df['league'] == "D"]['team']
league_D_number = len(league_D)
print(f"Teams in league A:", league_A_number),
print(f"Teams in league B:", league_B_number)
print(f"Teams in league C:", league_C_number)
print(f"Teams in league D:", league_D_number)

Teams in league A: 30
Teams in league B: 172
Teams in league C: 277
Teams in league D: 103


## Determination of odds per league

Here we set the betting odds for each league, which will allow us to draw final conclusions and establish the basic odds for individual teams.

> Remember: After generating the results, it is worth checking if they are reasonable.

In [122]:
league_mean = df.groupby('league')['victory_percentage'].mean()
print(f"League averages in winning percentages: \n ", league_mean)


League averages in winning percentages: 
  league
A    0.642500
B    0.551384
C    0.439686
D    0.300505
Name: victory_percentage, dtype: float64


In [115]:
odds_by_league = league_mean.apply(get_betting_odds)

In [123]:
print(f"Odds for individual leagues: \n ", odds_by_league)

Odds for individual leagues: 
  league
A    1.556420
B    1.813619
C    2.274351
D    3.327733
Name: victory_percentage, dtype: float64


# Discussion

We have obtained certain odds values for each league. But how does this translate into real business? The entire task was about determining certain values from which a bookmaker can begin operations. Correct determination of these values is critical to attract customers to place bets with us, and on the other hand, inappropriate determination may lead to financial losses in the first days of operation.

For this reason, before translating the results and recommendations into business objectives, the analysis is subjected to discussion. Therefore, we will now take on a review role and would like to verify the steps. To that end, we will collectively discuss and critique our work by answering the following questions together:
- What elements of the analysis were simplified? What was omitted in the analysis?
- Are there any inconsistencies in the estimated odds? What are they?
- How can we improve the odds estimates?
- How can we enrich our initial dataset to make the estimates more accurate and less risky?
- How can we simulate the outcomes of our analysis to verify that they do not lead to financial losses?

This is a discussion panel, and every idea is valuable here.
